# SQL Interview Questions — Senior Manager

Three questions for a Senior Manager in Data Science or Analytics. Allow **10–15 minutes per question**.

This notebook contains problem statements, table schemas, SQL answers, assessment points, and follow-up questions. SQL uses **PostgreSQL syntax** and is presented in Markdown for review; no database connection or package installation is required to read it. To execute a query, use a PostgreSQL database containing the specified tables and data.

Assume primary IDs are unique and non-null, foreign keys are valid, and timestamps use one agreed reporting timezone.

| Question | Topic |
|---|---|
| 1 | Aggregation, ranking, and ties |
| 2 | Monthly customer retention |
| 3 | Historical account status |

## Question 1: Find the top customers in each region

### Problem statement

You have two tables:

```sql
customers
---------
customer_id    INTEGER
customer_name  VARCHAR
region         VARCHAR

transactions
------------
transaction_id    INTEGER
customer_id       INTEGER
transaction_date  TIMESTAMP
amount            DECIMAL
status            VARCHAR
```

Find the **top three customers in each region by total completed transaction amount during September 2026**.

Requirements:
- Include only transactions with `status = 'completed'`.
- Use the **top three distinct total amounts**, including all customers tied at those ranks.
- Return region, customer ID, customer name, total amount, and rank.
- Only include customers with at least one qualifying transaction.

### Answer

```sql
WITH customer_totals AS (
    SELECT
        c.region,
        c.customer_id,
        c.customer_name,
        SUM(t.amount) AS total_amount
    FROM customers AS c
    JOIN transactions AS t
        ON c.customer_id = t.customer_id
    WHERE t.status = 'completed'
      AND t.transaction_date >= DATE '2026-09-01'
      AND t.transaction_date <  DATE '2026-10-01'
    GROUP BY
        c.region,
        c.customer_id,
        c.customer_name
),
ranked_customers AS (
    SELECT
        region,
        customer_id,
        customer_name,
        total_amount,
        DENSE_RANK() OVER (
            PARTITION BY region
            ORDER BY total_amount DESC
        ) AS customer_rank
    FROM customer_totals
)
SELECT
    region,
    customer_id,
    customer_name,
    total_amount,
    customer_rank
FROM ranked_customers
WHERE customer_rank <= 3
ORDER BY
    region,
    customer_rank,
    customer_id;
```

### What this tests

- Aggregation before ranking.
- Window functions with partitions.
- Handling ties.
- Correct timestamp boundaries.

### Senior-level follow-up

**What would change if the business wanted exactly three customers per region?**

Use `ROW_NUMBER()` with a deterministic tie-breaker, such as `customer_id`. This returns three customers when at least three qualify, otherwise all qualifying customers.

Clarify whether “top three” means the top three distinct totals (`DENSE_RANK`) or the first three customer positions with ties (`RANK`). For totals 100, 100, 90, 80, `DENSE_RANK` gives 1, 1, 2, 3; `RANK` gives 1, 1, 3, 4.

## Question 2: Calculate monthly customer retention

### Problem statement

You have a table containing customer activity:

```sql
customer_activity
-----------------
activity_id    INTEGER
customer_id    INTEGER
activity_date  TIMESTAMP
```

A customer is **active in a month if they have at least one activity record that month**.

For every month from **January through June 2026**, return:
- Month.
- Number of active customers.
- Number of those customers who were also active in the following calendar month.
- Next-month retention percentage.

For example, if January has 100 active customers and 60 of those customers are active in February, January's retention is 60%.

Assume data is complete through July 31, 2026. Include months with no activity and return `NULL` retention when the denominator is zero.

### Answer

```sql
WITH months AS (
    SELECT
        month_start::DATE AS activity_month
    FROM GENERATE_SERIES(
        DATE '2026-01-01',
        DATE '2026-06-01',
        INTERVAL '1 month'
    ) AS m(month_start)
),
monthly_activity AS (
    -- Each customer appears only once per month.
    SELECT DISTINCT
        customer_id,
        DATE_TRUNC('month', activity_date)::DATE AS activity_month
    FROM customer_activity
    WHERE activity_date >= DATE '2026-01-01'
      AND activity_date <  DATE '2026-08-01'
)
SELECT
    m.activity_month,
    COUNT(current_month.customer_id) AS active_customers,
    COUNT(next_month.customer_id) AS retained_customers,
    ROUND(
        100.0 * COUNT(next_month.customer_id)
        / NULLIF(COUNT(current_month.customer_id), 0),
        2
    ) AS retention_pct
FROM months AS m
LEFT JOIN monthly_activity AS current_month
    ON current_month.activity_month = m.activity_month
LEFT JOIN monthly_activity AS next_month
    ON next_month.customer_id = current_month.customer_id
   AND next_month.activity_month =
       current_month.activity_month + INTERVAL '1 month'
GROUP BY m.activity_month
ORDER BY m.activity_month;
```

### What this tests

- Defining a business metric correctly.
- Deduplicating customers at the correct level.
- Joining consecutive calendar months.
- Handling missing months and zero denominators.
- Avoiding integer division.

### Senior-level follow-up

**Why might retention for the latest month look unusually low?**

The following month may be incomplete, or data may arrive late. The metric needs an observation cutoff and a clear completeness rule.

## Question 3: Find each account's latest status correctly

### Problem statement

You have two tables:

```sql
accounts
--------
account_id   INTEGER
customer_id  INTEGER

account_status_history
----------------------
status_event_id  INTEGER
account_id       INTEGER
status           VARCHAR
effective_at    TIMESTAMP
```

An account can have multiple status changes, including being closed and later reopened.

Find all customers who had **at least two open accounts as of the end of September 30, 2026**.

Requirements:
- Determine each account's latest status before October 1, 2026.
- If two events have the same `effective_at`, the event with the larger `status_event_id` wins.
- An account with no status event before the cutoff is excluded.
- Return customer ID and number of open accounts.

Assume the account-to-customer mapping is stable.

### Answer

```sql
WITH ranked_statuses AS (
    SELECT
        account_id,
        status,
        ROW_NUMBER() OVER (
            PARTITION BY account_id
            ORDER BY
                effective_at DESC,
                status_event_id DESC
        ) AS status_rank
    FROM account_status_history
    WHERE effective_at < TIMESTAMP '2026-10-01 00:00:00'
),
open_accounts AS (
    SELECT
        account_id
    FROM ranked_statuses
    WHERE status_rank = 1
      AND status = 'open'
)
SELECT
    a.customer_id,
    COUNT(*) AS open_account_count
FROM accounts AS a
JOIN open_accounts AS o
    ON a.account_id = o.account_id
GROUP BY a.customer_id
HAVING COUNT(*) >= 2
ORDER BY
    open_account_count DESC,
    a.customer_id;
```

### What this tests

- Reconstructing state at a historical cutoff.
- Deterministic deduplication.
- Correct ordering of filtering and ranking.
- Distinguishing `WHERE` from `HAVING`.

### Senior-level follow-up

**Why can't we filter to `status = 'open'` before applying `ROW_NUMBER()`?**

That would find the most recent *open event*, rather than the most recent *status*. An account that subsequently closed would incorrectly be counted as open.

## Interviewer guidance

For a **Senior Manager**, assess both query correctness and whether the candidate clarifies business definitions, identifies data-quality risks, and explains the result clearly. Exact syntax recall should carry less weight than sound reasoning.

Accept alternative SQL solutions that satisfy the requirements. The examples above are reference answers, not the only valid approaches.